<a href="https://colab.research.google.com/github/jefferyocran/FraudGuard-OXGBoost/blob/main/experiment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U xgboost -q

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import recall_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
print("Ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 MB 4.8 MB/s eta 0:00:00
Ready.


In [3]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
uci = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])
uci['label'] = uci['label'].map({'ham': 0, 'spam': 1})
uci['source'] = 'uci'

field = pd.read_csv('ghana_momo_field.csv')[['label', 'message']]
field['source'] = 'field'

print("UCI:", len(uci), "| Field:", len(field))


UCI: 5572 | Field: 208


In [4]:
field_train, field_test = train_test_split(
    field, test_size=0.4, stratify=field['label'], random_state=SEED
)
print("Field train:", len(field_train), "| Field test:", len(field_test))


Field train: 124 | Field test: 84


In [5]:
train_A = uci.copy()                                    # UCI only
train_B = pd.concat([uci, field_train], ignore_index=True)  # UCI + Ghana

print("Condition A:", len(train_A), "| Condition B:", len(train_B))


Condition A: 5572 | Condition B: 5696


In [6]:
def train_and_test(train_df, test_df, name, field_boost=1):
    vec = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
    X_train = vec.fit_transform(train_df['message'])
    y_train = train_df['label']
    X_test = vec.transform(test_df['message'])
    y_test = test_df['label']

    # Give Ghanaian rows more weight so they aren't drowned by UCI
    weights = np.where(train_df['source'] == 'field', field_boost, 1)

    model = xgb.XGBClassifier(
        max_depth=6, n_estimators=200, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, eval_metric='logloss'
    )
    model.fit(X_train, y_train, sample_weight=weights)
    preds = model.predict(X_test)

    scam_recall = recall_score(y_test, preds, pos_label=1)
    print(f"\n=== {name} ===")
    print(f"Ghanaian SCAM recall: {round(scam_recall*100, 1)}%")
    print(confusion_matrix(y_test, preds))
    return scam_recall

recall_A = train_and_test(train_A, field_test, "Condition A: UCI only", field_boost=1)
recall_B = train_and_test(train_B, field_test, "Condition B: UCI + Ghana (weighted)", field_boost=100)

print("\n----------------------------------------")
print(f"Without local data:       {round(recall_A*100,1)}%")
print(f"With weighted local data: {round(recall_B*100,1)}%")
print(f"Improvement:              {round((recall_B-recall_A)*100,1)} pp")



=== Condition A: UCI only ===
Ghanaian SCAM recall: 23.5%
[[39 28]
 [13  4]]

=== Condition B: UCI + Ghana (weighted) ===
Ghanaian SCAM recall: 29.4%
[[63  4]
 [12  5]]

----------------------------------------
Without local data:       23.5%
With weighted local data: 29.4%
Improvement:              5.9 pp


In [7]:
# Experiment 3 — gentler weighting (boost=3)
def focal_loss_objective(y_pred, dtrain):
    gamma, alpha = 2.0, 0.25
    y_true = dtrain.get_label()
    p = 1.0 / (1.0 + np.exp(-y_pred))
    p_t = y_true * p + (1 - y_true) * (1 - p)
    alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
    fw = alpha_t * np.power(1 - p_t, gamma)
    grad = fw * (p - y_true)
    hess = fw * p * (1 - p) * (gamma * (1 - p_t) + 1)
    return grad, hess

vec = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
X_train = vec.fit_transform(train_B['message'])
X_test = vec.transform(field_test['message'])
y_train = train_B['label']
y_test = field_test['label']

BOOST = 10          # gentler — was 15
weights = np.where(train_B['source'] == 'field', BOOST, 1)
print("Using boost =", BOOST)

dtrain = xgb.DMatrix(X_train, label=y_train, weight=weights)
dtest = xgb.DMatrix(X_test, label=y_test)
params = {'max_depth':6,'eta':0.1,'subsample':0.8,'colsample_bytree':0.8,'seed':SEED}

std_model = xgb.train({**params,'objective':'binary:logistic'}, dtrain, 200)
o_model   = xgb.train(params, dtrain, 200, obj=focal_loss_objective)

for model, name in [(std_model,"Standard XGBoost"), (o_model,"O-XGBoost (focal)")]:
    probs = 1.0/(1.0+np.exp(-model.predict(dtest)))
    preds = (probs>=0.5).astype(int)
    print(f"\n=== {name} + local data ===")
    print("Scam recall:", round(recall_score(y_test,preds,pos_label=1)*100,1),"%")
    print(confusion_matrix(y_test,preds))

Using boost = 10

=== Standard XGBoost + local data ===
Scam recall: 100.0 %
[[ 0 67]
 [ 0 17]]

=== O-XGBoost (focal) + local data ===
Scam recall: 5.9 %
[[61  6]
 [16  1]]


In [8]:
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score

def make_focal(gamma, alpha=0.25):
    def obj(y_pred, dtrain):
        y_true = dtrain.get_label()
        p = 1.0/(1.0+np.exp(-y_pred))
        p_t = y_true*p + (1-y_true)*(1-p)
        a_t = y_true*alpha + (1-y_true)*(1-alpha)
        fw = a_t*np.power(1-p_t, gamma)
        grad = fw*(p-y_true)
        hess = fw*p*(1-p)*(gamma*(1-p_t)+1)
        return grad, hess
    return obj

uci2 = uci.copy(); uci2['source']='uci'
field2 = field.copy(); field2['source']='field'
f_tr, f_te = train_test_split(field2, test_size=0.4, stratify=field2['label'], random_state=SEED)
train_B = pd.concat([uci2, f_tr], ignore_index=True)

vec = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
Xtr = vec.fit_transform(train_B['message']); ytr = train_B['label']
Xte = vec.transform(f_te['message']);        yte = f_te['label']

w = np.where(train_B['source']=='field', 6, 1)
dtr = xgb.DMatrix(Xtr, label=ytr, weight=w)
dte = xgb.DMatrix(Xte, label=yte)
params = {'max_depth':6,'eta':0.1,'subsample':0.8,'colsample_bytree':0.8,'seed':SEED}

print("gamma | recall | precision | scams caught")
print("-"*45)
for g in [2, 3, 5]:
    m = xgb.train(params, dtr, 200, obj=make_focal(g))
    probs = 1.0/(1.0+np.exp(-m.predict(dte)))
    preds = (probs>=0.3).astype(int)
    r = recall_score(yte, preds, pos_label=1, zero_division=0)
    p = precision_score(yte, preds, pos_label=1, zero_division=0)
    caught = int(((preds==1)&(yte==1)).sum()); tot = int((yte==1).sum())
    print(f"  {g}   | {r*100:5.1f}% |  {p*100:5.1f}%  |   {caught}/{tot}")

gamma | recall | precision | scams caught
---------------------------------------------
  2   |  52.9% |   33.3%  |   9/17
  3   |  70.6% |   27.3%  |   12/17
  5   |  94.1% |   20.0%  |   16/17
